# 150 — Assemble Falcon 9 release 1.0.0

Code and release metadata come from config.PROJECT_ROOT; observations from config.INPUT_DIR; products from config.OUTPUT_DIR. Audits go to config.NB150_DIR and the package to config.RELEASE_DIR. The raw accident window follows config.RAW_MSEED. Extended 48-hour/precursor records are excluded.

Missing required payloads stop the build by default. Optional supporting files and disclosed scientific/reproducibility limitations do not block assembly; assembly does not certify readiness for public upload. No upload is performed.


Defaults select main61.tex, supplement61.tex and falcon9_bibfile.bib from PROJECT_ROOT/latex. Override using ZENODO_MANUSCRIPT_FILES if needed. Set ZENODO_BUILD=0 for audit only. Required payload files stop a build by default; optional calibration/video provenance and disclosed rerun limitations do not. No upload is performed. The 48-hour/precursor records are excluded.


In [1]:
from __future__ import annotations

import csv
import dataclasses
import hashlib
import json
import os
import re
import shutil
import sys
import tempfile
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "modules" / "project_config.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate modules/project_config.py")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from modules import project_config as config
if Path(config.__file__).resolve() != PROJECT_ROOT / "modules" / "project_config.py":
    raise RuntimeError("Wrong project_config loaded; restart the kernel in this repository")

INPUT_ROOT = Path(config.INPUT_DIR).resolve()
OUTPUT_ROOT = Path(config.OUTPUT_DIR).resolve()
RELEASE_PARENT = Path(config.RELEASE_DIR).resolve()
AUDIT_ROOT = Path(config.NB150_DIR).resolve()
RELEASE_NAME = os.environ.get("ZENODO_RELEASE_NAME", "falcon9_seismoacoustic_release")
if (not RELEASE_NAME or RELEASE_NAME in {".", ".."}
        or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_." for c in RELEASE_NAME)):
    raise ValueError("ZENODO_RELEASE_NAME must be a single safe directory name")
# The shared config owns the release destination, including any machine-specific path.
if os.environ.get("ZENODO_RELEASE_PARENT"):
    if Path(os.environ["ZENODO_RELEASE_PARENT"]).expanduser().resolve() != RELEASE_PARENT:
        raise ValueError("Set the release destination through config.RELEASE_DIR; remove ZENODO_RELEASE_PARENT")
RELEASE_ROOT = RELEASE_PARENT / RELEASE_NAME
ZIP_FILE = RELEASE_PARENT / f"{RELEASE_NAME}.zip"

# Select exact manuscript versions explicitly; no dated filenames are inferred.
# Example: ZENODO_MANUSCRIPT_FILES='["main59.tex", "supplement58.tex", "mybibfile52.bib"]'
MANUSCRIPT_ROOT = Path(os.environ.get("ZENODO_MANUSCRIPT_DIR")
    or os.environ.get("ZENODO_REVIEW_ASSETS") or PROJECT_ROOT / "latex").expanduser().resolve()
MANUSCRIPT_FILES = json.loads(os.environ.get("ZENODO_MANUSCRIPT_FILES",
    '["main61.tex", "supplement61.tex", "falcon9_bibfile.bib"]'))
if not isinstance(MANUSCRIPT_FILES, list) or not MANUSCRIPT_FILES or not all(isinstance(p, str) for p in MANUSCRIPT_FILES):
    raise ValueError("ZENODO_MANUSCRIPT_FILES must be a nonempty JSON list of relative filenames")

BUILD_RELEASE = os.environ.get("ZENODO_BUILD", "1") == "1"
REPLACE_EXISTING = os.environ.get("ZENODO_REPLACE", "0") == "1"
STRICT_REQUIRED_FILES = os.environ.get("ZENODO_STRICT", "1") == "1"
INCLUDE_CONVENIENCE_PDFS = os.environ.get("ZENODO_INCLUDE_PDFS", "1") == "1"
INCLUDE_REGIONAL_OBSERVATIONS = os.environ.get("ZENODO_INCLUDE_REGIONAL", "1") == "1"
INCLUDE_PERMISSION_DEPENDENT_VIDEO = os.environ.get("ZENODO_INCLUDE_VIDEO", "0") == "1"
MAKE_ZIP = os.environ.get("ZENODO_MAKE_ZIP", "1") == "1"

SOURCE_ROOTS = {"project": PROJECT_ROOT, "inputs": INPUT_ROOT,
                "outputs": OUTPUT_ROOT, "manuscript": MANUSCRIPT_ROOT}
DESTINATION_PREFIXES = {"project": "repository", "inputs": "paper/inputs",
                        "outputs": "paper/outputs", "manuscript": "manuscript"}
for root in SOURCE_ROOTS.values():
    if root == RELEASE_ROOT or RELEASE_ROOT in root.parents:
        raise ValueError(f"A source root cannot be inside the release being replaced: {root}")
print("Repository:", PROJECT_ROOT)
print("External inputs:", INPUT_ROOT)
print("External outputs:", OUTPUT_ROOT)
print("Release:", RELEASE_ROOT)
print("Audit reports:", AUDIT_ROOT)
print("Manuscript:", MANUSCRIPT_ROOT, MANUSCRIPT_FILES)
print("Build enabled:", BUILD_RELEASE)


Repository: /Users/thompsong/Developer/KSCRocketSeismology/08_falcon9_fireball_paper
External inputs: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/falcon9_fireball_paper/inputs
External outputs: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/falcon9_fireball_paper/outputs
Release: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/falcon9_fireball_paper/release/falcon9_seismoacoustic_release
Audit reports: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/falcon9_fireball_paper/outputs/150_assemble_zenodo_release
Manuscript: /Users/thompsong/Developer/KSCRocketSeismology/08_falcon9_fireball_paper/latex ['main61.tex', 'supplement61.tex', 'falcon9_bibfile.bib']
Build enabled: True


## File inventory

Rules declare their source root explicitly. Destinations preserve paths relative to that root under separate archive prefixes. Current notebook names and configured NB directories are used throughout. The private local config, object caches, frame caches, and event galleries are excluded. Raw/source video is excluded; rendered movies are opt-in.


In [2]:
@dataclasses.dataclass(frozen=True)
class Rule:
    label: str
    category: str
    patterns: tuple[str, ...]
    required: bool = True
    source_root: str = "project"
    note: str = ""


rules: list[Rule] = []
def add(label, category, *patterns, **kwargs):
    rules.append(Rule(label, category, tuple(patterns), **kwargs))

WORKFLOW = (
    (0, "correct_bchh_instrument_response"),
    (10, "prepare_analysis_inputs"),
    (20, "analyze_ksc_weather"),
    (30, "reconcile_manual_event_catalogues"),
    (40, "validate_event_catalogue_with_array_processing"),
    (50, "prepare_event_waveform_products"),
    (60, "analyze_planar_array_propagation"),
    (70, "measure_event_amplitudes_and_acoustic_seismic_coupling"),
    (80, "analyze_named_events_and_timing"),
    (90, "estimate_acoustic_energetics"),
    (100, "analyze_seismic_polarization"),
    (110, "search_for_direct_seismic_waves"),
    (120, "analyze_regional_detectability"),
    (130, "generate_supplement_event_table"),
    (140, "generate_publication_figures"),
    (150, "assemble_zenodo_release"),
)
add("Repository instructions", "code", "README*.md")
add("Environment specification", "code", "environment.yml", "environment.yaml", "requirements*.txt", "pyproject.toml")
add("Software license", "code", "LICENSE", "LICENSE.*")
add("Citation metadata", "code", "CITATION.cff")
add("Data manifest", "documentation", "DATA_MANIFEST.md")
add("Local config template", "code", "local_config.example.toml")
add("Python modules", "code", "modules/**/*.py")
for number, slug in WORKFLOW:
    add(f"Notebook {number:03d}", "code", f"notebooks/{number:03d}_{slug}.ipynb")
add("Validation notebooks", "code", "notebooks/validation/*.ipynb", required=False)

# Each required observation is checked separately, not hidden by a broad glob.
for label, path in (
    ("Raw six-channel accident waveform", Path(config.RAW_MSEED).resolve().relative_to(INPUT_ROOT).as_posix()),
    ("Original BCHH response metadata", "metadata/KSC.xml"),
    ("NASA/KSC weather workbook", "metadata/nasa_weather_tower_data.xls"),
    ("Launch-pad and camera coordinates", "metadata/launchpads_cameras.kml"),
    ("CSS calibration", "metadata/sitedb.calibration"),
    ("Reviewed BCHH arrival picks", "metadata/bchh_named_event_pressure_arrival_picks.csv"),
    ("UCS-3 event frames", "metadata/ucs3_video_event_frames.csv"),
):
    add(label, "observations", path, source_root="inputs")
for filename in ("sitedb.site", "sitedb.sitechan"):
    add(filename, "metadata", f"metadata/{filename}", source_root="inputs", required=False)
for filename in ("manual_arrival_picks.csv", "legacy_infrasound_event_catalogue.csv", "legacy_catalog_157_events.csv", "legacy_export_summary.txt"):
    add(filename, "observations", f"legacy_export/{filename}", source_root="inputs")
# Extended 48-hour/precursor records are intentionally outside this release.
add("Calibration provenance", "documentation", "documentation/calibration_provenance.md", "docs/calibration_provenance.md", required=False)
add("Calibration comparisons", "open_evidence", "calibration_checks/**/*.csv", "calibration_checks/**/*.json", "calibration_checks/**/*.md",
    source_root="inputs", required=False)

# Resolve each product through the current config, including customized NB paths.
def product_pattern(number, tail):
    directory = Path(getattr(config, f"NB{number:03d}_DIR")).resolve()
    try:
        relative = directory.relative_to(OUTPUT_ROOT)
    except ValueError as exc:
        raise ValueError(f"NB{number:03d}_DIR must lie under config.OUTPUT_DIR") from exc
    return (relative / tail).as_posix()

def product(label, number, *tails, category="derived", required=True):
    add(label, category, *(product_pattern(number, tail) for tail in tails),
        source_root="outputs", required=required)

for number, slug in WORKFLOW[:-1]:
    product(f"{number:03d} compact products", number, "*.csv", "*.json", "*.xml", "*.mseed", required=False)
    if INCLUDE_CONVENIENCE_PDFS:
        product(f"{number:03d} root figures", number, "*.pdf", category="figures", required=False)
for number, subdir in ((10, "derived"), (50, "derived"), (100, "multiband")):
    product(f"{number:03d} nested tables", number, f"{subdir}/*.csv", f"{subdir}/*.json", required=False)
for label, number, filename in (
    ("Analysis configuration", 10, "analysis_configuration.json"),
    ("Weather summary", 20, "weather_acoustic_summary.csv"),
    ("Accepted event catalogue", 30, "final_153_event_catalogue.csv"),
    ("Amplitude measurements", 70, "event_acoustic_seismic_amplitudes.csv"),
    ("Named-event timing comparison", 80, "named_event_observed_vs_modeled_timing.csv"),
    ("Supplement event table", 130, "supplement_event_measurements.tex"),
    ("Supplement table provenance", 130, "supplement_event_measurements.json"),
):
    product(label, number, filename)
product("Video synchronization provenance", 140, "video_sync_configuration.json", required=False)
if INCLUDE_CONVENIENCE_PDFS:
    for number, subdir in ((10, "summary_figures"), (20, "figures"), (30, "figures"),
                           (40, "figures"), (60, "composites"), (70, "figures"),
                           (100, "multiband"), (110, "figures")):
        product(f"{number:03d} summary figures", number, f"{subdir}/*.pdf", category="figures", required=False)
if INCLUDE_REGIONAL_OBSERVATIONS:
    product("Regional MiniSEED records", 120, "waveform_cache/**/*.mseed", category="regional_observations")
    product("Regional response StationXML", 120, "response_cache/**/*.xml", category="regional_metadata")
    product("Regional event inventory", 120, "regional_station_inventory.xml", category="regional_metadata")
for filename in MANUSCRIPT_FILES:
    add(f"Manuscript {filename}", "manuscript", filename, source_root="manuscript")
add("Manuscript support", "manuscript", "*.cls", "*.sty", "*.bst", "banner.png", "figures/*.pdf", "generated/*.tex",
    source_root="manuscript", required=False)
if INCLUDE_PERMISSION_DEPENDENT_VIDEO:
    product("Authorized rendered movies", 140, "*.mp4", "*.mov", category="permission_dependent")

# Review these against current evidence, then explicitly mark resolved=True.
# Historical counts and manuscript version numbers are deliberately not assumed.
author_actions = [
    {"id": "scope", "resolved": True, "action": "Extended 48-hour/precursor records excluded by author; no claim to reproduce that search."},
    {"id": "regional", "resolved": False, "action": "Retain uncertainty in regional attribution and record the prior visual review; no automatic detection claim."},
    {"id": "licenses", "resolved": False, "action": "Before upload, confirm redistribution rights and attribution for included observations and third-party assets."},
    {"id": "portable_rerun", "resolved": False, "action": "Clean rerun not certified; disclose absolute paths and independently obtained video requirements."},
    {"id": "doi", "resolved": False, "action": "Add Zenodo DOI when assigned; a reserved DOI is not required to assemble locally."},
]

if INCLUDE_PERMISSION_DEPENDENT_VIDEO:
    author_actions.append({"id": "video_permission", "resolved": False,
        "action": "Confirm redistribution permission for every selected movie; inclusion does not establish permission."})
print(f"Declared {len(rules)} file rules and {len(author_actions)} review actions")


Declared 95 file rules and 5 review actions


## Resolve and audit

Require safe relative patterns, reject symlink escapes, deduplicate overlapping rules, and reject destination collisions. Missing required files are listed separately from optional products.


In [3]:
def safe_relative(value):
    path = Path(value)
    if not value or path.is_absolute() or ".." in path.parts or "\\" in value:
        raise ValueError(f"Expected a safe relative path: {value!r}")
    return path


def resolve_rule(rule):
    if rule.source_root not in SOURCE_ROOTS:
        raise ValueError(f"Unknown source root: {rule.source_root}")
    root = SOURCE_ROOTS[rule.source_root]
    matches = {}
    for pattern in rule.patterns:
        safe_relative(pattern)
        for path in sorted(root.glob(pattern)):
            if not path.is_file():
                continue
            # Never package extended precursor/48-hour MiniSEED, even via another rule.
            name_lower = path.name.lower()
            if path.suffix.lower() in {".mseed", ".miniseed"} and (
                "precursor" in name_lower or re.search(r"48[ _-]*(?:h|hour)", name_lower)
            ):
                continue
            resolved = path.resolve()
            if not resolved.is_relative_to(root):
                raise ValueError(f"Source symlink escapes its declared root: {path}")
            if resolved == RELEASE_ROOT or resolved.is_relative_to(RELEASE_ROOT):
                raise ValueError(f"Release cannot include itself: {path}")
            matches[path.relative_to(root).as_posix()] = path
    return sorted(matches.items())


resolved_rows = []
selected = {}
missing_required = []
for rule in rules:
    matches = resolve_rule(rule)
    if not matches and rule.required:
        missing_required.append(rule)
    resolved_rows.append({"label": rule.label, "category": rule.category,
        "source_root": rule.source_root, "required": rule.required,
        "match_count": len(matches), "status": "FOUND" if matches else ("MISSING" if rule.required else "OPTIONAL_ABSENT"),
        "patterns": list(rule.patterns), "note": rule.note})
    for relative, source in matches:
        destination = (Path(DESTINATION_PREFIXES[rule.source_root]) / safe_relative(relative)).as_posix()
        previous = selected.get(destination)
        if previous and previous[0].resolve() != source.resolve():
            raise RuntimeError(f"Destination collision: {destination}")
        selected.setdefault(destination, (source, rule))
for destination, (source, rule) in selected.items():
    if source.name == "local_config.toml" or source.suffix.lower() in {".pkl", ".pickle", ".npz"}:
        raise RuntimeError(f"Private config or object cache selected: {source}")
    if set(source.parts) & {"frames_phase1", "event_diagnostics", "event_figures", "surfaces", "__pycache__"}:
        raise RuntimeError(f"Diagnostic/cache directory selected: {source}")
print(f"Selected {len(selected):,} unique files ({sum(p.stat().st_size for p, _ in selected.values()) / 1e6:.1f} MB)")
print(f"Missing required rules: {len(missing_required)}")
for rule in missing_required:
    print("  -", rule.label, "=>", rule.source_root, rule.patterns)


Selected 660 unique files (249.1 MB)
Missing required rules: 0


## Assemble and verify

Audit reports are always written to `config.NB150_DIR`. Set `ZENODO_STRICT=1` to stop on missing files. Builds stage and verify payloads before installation. Existing directories or ZIPs require `ZENODO_REPLACE=1`; previous artifacts are retained as timestamped backups and restored if installation fails. With `ZENODO_MAKE_ZIP=0`, an existing ZIP is also backed up so it cannot be mistaken for the new build.


In [4]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_payload(stage, manifest):
    for row in manifest:
        path = stage / safe_relative(row["path"])
        if (not path.is_file() or path.is_symlink() or not path.resolve().is_relative_to(stage.resolve())
                or path.stat().st_size != row["size_bytes"] or sha256_file(path) != row["sha256"]):
            raise RuntimeError(f"Manifest verification failed: {row['path']}")


def write_reports(directory, manifest):
    directory.mkdir(parents=True, exist_ok=True)
    pending = [a for a in author_actions if not a["resolved"]]
    status = "INCOMPLETE" if missing_required else "ASSEMBLED_FOR_UPLOAD_REVIEW"
    for filename, payload in (("file_manifest.json", manifest), ("inventory_rule_status.json", resolved_rows),
                              ("outstanding_author_actions.json", author_actions)):
        (directory / filename).write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    with (directory / "file_manifest.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=["path", "category", "size_bytes", "sha256", "source_rule"])
        writer.writeheader()
        writer.writerows(manifest)
    lines = ["# Falcon 9 release assembly", "", f"Generated: {datetime.now(timezone.utc).isoformat()}",
             f"Status: **{status}**", f"Build requested: {BUILD_RELEASE}", f"Selected payload files: {len(manifest)}", "",
             "## Missing required files", ""]
    lines += [f"- {r.label} ({r.source_root}): {', '.join(r.patterns)}" for r in missing_required] or ["- None"]
    lines += ["", "## Pre-upload checks and disclosed limitations (not missing files)", ""]
    lines += [f"- {a['id']}: {a['action']}" for a in pending] or ["- None"]
    lines += ["", "## Package layout and reuse", "",
        "Code is under repository/; external data and products are under paper/inputs/ and paper/outputs/.",
        "Manuscript source is under manuscript/. Create repository/local_config.toml from its example,",
        'then set [external_paths] paper_dir = "../paper" (resolved relative to repository/).',
        "The source video and SDS archive are not included. Configure separately when needed.",
        "Existing product JSON/CSV files and saved notebook outputs may contain the original machine's absolute paths.",
        "Regenerate upstream products on the new machine; this assembler preserves scientific source bytes.",
        "The generated supplement table is under paper/outputs/130_generate_supplement_event_table/.",
        "Update manuscript input paths or copy the generated table into the manuscript build workspace before compiling.",
        "Saved notebook outputs are historical; a package build is not a scientific rerun.",
        "Pickles, NPZ surfaces, event galleries, frame caches and private local_config.toml are excluded.",
        "The payload manifest covers selected source files. SHA256SUMS covers those plus generated reports.", ""]
    (directory / "ASSEMBLY_STATUS.md").write_text("\n".join(lines), encoding="utf-8")
    return status


manifest_rows = [{"path": destination, "category": rule.category,
    "size_bytes": source.stat().st_size, "sha256": sha256_file(source), "source_rule": rule.label}
    for destination, (source, rule) in sorted(selected.items())]
# Audits always live with Notebook 150 products, including strict-mode failures.
release_status = write_reports(AUDIT_ROOT, manifest_rows)
if STRICT_REQUIRED_FILES and missing_required:
    raise FileNotFoundError(f"Required files are missing; see {AUDIT_ROOT / 'ASSEMBLY_STATUS.md'}")

if BUILD_RELEASE:
    RELEASE_PARENT.mkdir(parents=True, exist_ok=True)
    # Check both artifacts, even if ZIP creation is disabled, to avoid stale ZIPs.
    existing = [p for p in (RELEASE_ROOT, ZIP_FILE) if p.exists() or p.is_symlink()]
    if existing and not REPLACE_EXISTING:
        raise FileExistsError(f"Existing release artifacts: {existing}; set ZENODO_REPLACE=1 intentionally")
    temporary = Path(tempfile.mkdtemp(prefix=f".{RELEASE_NAME}-", dir=RELEASE_PARENT))
    temporary_zip = RELEASE_PARENT / f".{RELEASE_NAME}-{uuid.uuid4().hex}.zip.tmp"
    backups = []
    installed = []
    try:
        for row in manifest_rows:
            source, _ = selected[row["path"]]
            target = temporary / row["path"]
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
        verify_payload(temporary, manifest_rows)
        write_reports(temporary / "documentation", manifest_rows)
        checksums = {p.relative_to(temporary).as_posix(): sha256_file(p)
                     for p in sorted(temporary.rglob("*")) if p.is_file()}
        (temporary / "SHA256SUMS").write_text("".join(f"{digest}  {name}\n" for name, digest in checksums.items()), encoding="utf-8")
        if MAKE_ZIP:
            with zipfile.ZipFile(temporary_zip, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
                for path in sorted(temporary.rglob("*")):
                    if path.is_file():
                        archive.write(path, Path(RELEASE_NAME) / path.relative_to(temporary))
            with zipfile.ZipFile(temporary_zip) as archive:
                if archive.testzip() is not None:
                    raise RuntimeError("ZIP integrity check failed")
        # Preserve previous artifacts; restore them if installation fails.
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:8]
        for old in existing:
            backup = old.with_name(old.name + ".backup-" + stamp)
            old.replace(backup)
            backups.append((old, backup))
        temporary.replace(RELEASE_ROOT)
        installed.append(RELEASE_ROOT)
        if MAKE_ZIP:
            temporary_zip.replace(ZIP_FILE)
            installed.append(ZIP_FILE)
    except Exception:
        for path in reversed(installed):
            if path.is_dir() and not path.is_symlink():
                shutil.rmtree(path)
            else:
                path.unlink()
        for old, backup in reversed(backups):
            backup.replace(old)
        raise
    finally:
        if temporary.exists():
            shutil.rmtree(temporary)
        if temporary_zip.exists():
            temporary_zip.unlink()
    print("Staging directory:", RELEASE_ROOT)
    if MAKE_ZIP:
        print("ZIP:", ZIP_FILE)
    for _, backup in backups:
        print("Previous artifact retained:", backup)
    print("Release status:", release_status)
else:
    print("Audit only. Reports written to:", AUDIT_ROOT)


Staging directory: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/falcon9_fireball_paper/release/falcon9_seismoacoustic_release
ZIP: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/falcon9_fireball_paper/release/falcon9_seismoacoustic_release.zip
Release status: ASSEMBLED_FOR_UPLOAD_REVIEW


## Final verification

Verify payload hashes, generated-report checksums, ZIP integrity, and absence of symlinks. This does not certify scientific review, licensing, DOI metadata, or an end-to-end rerun. Embedded old absolute paths must be regenerated on the receiving machine.


In [5]:
if BUILD_RELEASE:
    manifest = json.loads((RELEASE_ROOT / "documentation/file_manifest.json").read_text())
    verify_payload(RELEASE_ROOT, manifest)
    for line in (RELEASE_ROOT / "SHA256SUMS").read_text().splitlines():
        digest, relative = line.split("  ", 1)
        path = RELEASE_ROOT / safe_relative(relative)
        if path.is_symlink() or not path.resolve().is_relative_to(RELEASE_ROOT.resolve()) or sha256_file(path) != digest:
            raise RuntimeError(f"Package checksum mismatch: {relative}")
    if any(p.is_symlink() for p in RELEASE_ROOT.rglob("*")):
        raise RuntimeError("Unexpected symlink in release")
    if MAKE_ZIP:
        with zipfile.ZipFile(ZIP_FILE) as archive:
            if archive.testzip() is not None:
                raise RuntimeError("ZIP integrity check failed")
    print(f"Verified {len(manifest)} payload files and all generated report checksums")
    print((RELEASE_ROOT / "documentation/ASSEMBLY_STATUS.md").read_text())


Verified 660 payload files and all generated report checksums
# Falcon 9 release assembly

Generated: 2026-10-06T23:45:46.142715+00:00
Status: **ASSEMBLED_FOR_UPLOAD_REVIEW**
Build requested: True
Selected payload files: 660

## Missing required files

- None

## Pre-upload checks and disclosed limitations (not missing files)

- regional: Retain uncertainty in regional attribution and record the prior visual review; no automatic detection claim.
- licenses: Before upload, confirm redistribution rights and attribution for included observations and third-party assets.
- portable_rerun: Clean rerun not certified; disclose absolute paths and independently obtained video requirements.
- doi: Add Zenodo DOI when assigned; a reserved DOI is not required to assemble locally.

## Package layout and reuse

Code is under repository/; external data and products are under paper/inputs/ and paper/outputs/.
Manuscript source is under manuscript/. Create repository/local_config.toml from its example,
